In [4]:
from google.colab import files

uploaded = files.upload()

Saving fake_job_postings.csv to fake_job_postings.csv


In [5]:
import pandas as pd

df = pd.read_csv("fake_job_postings.csv")

print(df.shape)

(17880, 18)


In [6]:
text_columns = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits"
]

df["combined_text"] = (
    df[text_columns]
    .fillna("")
    .agg(" ".join, axis=1)
)

print(df["combined_text"].iloc[0])

Marketing Intern We're Food52, and we've created a groundbreaking and award-winning cooking site. We support, connect, and celebrate home cooks, and give them everything they need in one place.We have a top editorial, business, and engineering team. We're focused on using technology to find new and better ways to connect people around their specific food interests, and to offer them superb, highly curated information about food and cooking. We attract the most talented home cooks and contributors in the country; we also publish well-known professionals like Mario Batali, Gwyneth Paltrow, and Danny Meyer. And we have partnerships with Whole Foods Market and Random House.Food52 has been named the best food website by the James Beard Foundation and IACP, and has been featured in the New York Times, NPR, Pando Daily, TechCrunch, and on the Today Show.We're located in Chelsea, in New York City. Food52, a fast-growing, James Beard Award-winning online food community and crowd-sourced and cur

In [7]:
from sklearn.model_selection import train_test_split

X = df["combined_text"]
y = df["fraudulent"]

# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Split training portion into train + validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

Train: 11443
Validation: 2861
Test: 3576


In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Create tokenizer
tokenizer = Tokenizer(
    num_words=30000,
    oov_token="<OOV>"
)

# Learn vocabulary ONLY from training data
tokenizer.fit_on_texts(X_train)

# Convert text → integer sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Maximum sequence length
MAX_LEN = 500

# Pad sequences
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

print("Training shape   :", X_train_pad.shape)
print("Validation shape :", X_val_pad.shape)
print("Test shape       :", X_test_pad.shape)
print("Vocabulary size  :", len(tokenizer.word_index))

Training shape   : (11443, 500)
Validation shape : (2861, 500)
Test shape       : (3576, 500)
Vocabulary size  : 102137


In [10]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

VOCAB_SIZE = 30000

model_bilstm = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128
    ),

    Bidirectional(
        LSTM(64)
    ),

    Dropout(0.5),

    Dense(32, activation="relu"),

    Dropout(0.3),

    Dense(1, activation="sigmoid")
])

model_bilstm.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

model_bilstm.build(input_shape=(None, MAX_LEN))

model_bilstm.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,942,977 (15.04 MB)

 Trainable params: 3,942,977 (15.04 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weight_dict = dict(zip(classes, class_weights))

print("Class weights:")
print(class_weight_dict)

Class weights:
{np.int64(0): np.float64(0.5254385159335109), np.int64(1): np.float64(10.327617328519855)}


In [12]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model_bilstm.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=10,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/10
179/179 ━━━━━━━━━━━━━━━━━━━━ 245s 1s/step - accuracy: 0.8150 - loss: 0.4125 - precision: 0.1863 - recall: 0.8375 - val_accuracy: 0.9140 - val_loss: 0.2447 - val_precision: 0.3458 - val_recall: 0.8633
Epoch 2/10
179/179 ━━━━━━━━━━━━━━━━━━━━ 240s 1s/step - accuracy: 0.9613 - loss: 0.1147 - precision: 0.5579 - recall: 0.9657 - val_accuracy: 0.9713 - val_loss: 0.0852 - val_precision: 0.6707 - val_recall: 0.8058
Epoch 3/10
179/179 ━━━━━━━━━━━━━━━━━━━━ 234s 1s/step - accuracy: 0.9876 - loss: 0.0441 - precision: 0.8021 - recall: 0.9874 - val_accuracy: 0.9584 - val_loss: 0.1304 - val_precision: 0.5450 - val_recall: 0.8705
Epoch 4/10
179/179 ━━━━━━━━━━━━━━━━━━━━ 243s 1s/step - accuracy: 0.9939 - loss: 0.0187 - precision: 0.8891 - recall: 0.9982 - val_accuracy: 0.9619 - val_loss: 0.1485 - val_precision: 0.5750 - val_recall: 0.8273


In [13]:
test_results = model_bilstm.evaluate(
    X_test_pad,
    y_test,
    verbose=0
)

print("Test results:")
for name, value in zip(model_bilstm.metrics_names, test_results):
    print(f"{name}: {value:.4f}")

Test results:
loss: 0.0708
compile_metrics: 0.9773


In [14]:
y_prob_bilstm = model_bilstm.predict(
    X_test_pad,
    verbose=0
).ravel()

y_pred_bilstm = (y_prob_bilstm >= 0.5).astype(int)

print("Predictions generated.")

Predictions generated.


In [15]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

y_prob_bilstm = model_bilstm.predict(
    X_test_pad,
    verbose=0
).ravel()

y_pred_bilstm = (y_prob_bilstm >= 0.5).astype(int)

print("Accuracy :", accuracy_score(y_test, y_pred_bilstm))
print("Precision:", precision_score(y_test, y_pred_bilstm))
print("Recall   :", recall_score(y_test, y_pred_bilstm))
print("F1 Score :", f1_score(y_test, y_pred_bilstm))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_bilstm))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_bilstm))

Accuracy : 0.9773489932885906
Precision: 0.7149532710280374
Recall   : 0.884393063583815
F1 Score : 0.7906976744186046

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      3403
           1       0.71      0.88      0.79       173

    accuracy                           0.98      3576
   macro avg       0.85      0.93      0.89      3576
weighted avg       0.98      0.98      0.98      3576


Confusion Matrix:
[[3342   61]
 [  20  153]]


In [17]:
from sklearn.metrics import average_precision_score

pr_auc_bilstm = average_precision_score(
    y_test,
    y_prob_bilstm
)

print("BiLSTM PR-AUC:", pr_auc_bilstm)

BiLSTM PR-AUC: 0.873840439124773


In [18]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

for threshold in thresholds:
    y_pred_threshold = (y_prob_bilstm >= threshold).astype(int)

    print(
        f"Threshold: {threshold:.1f} | "
        f"Precision: {precision_score(y_test, y_pred_threshold):.3f} | "
        f"Recall: {recall_score(y_test, y_pred_threshold):.3f} | "
        f"F1: {f1_score(y_test, y_pred_threshold):.3f}"
    )

Threshold: 0.2 | Precision: 0.593 | Recall: 0.902 | F1: 0.716
Threshold: 0.3 | Precision: 0.647 | Recall: 0.902 | F1: 0.754
Threshold: 0.4 | Precision: 0.683 | Recall: 0.896 | F1: 0.775
Threshold: 0.5 | Precision: 0.715 | Recall: 0.884 | F1: 0.791
Threshold: 0.6 | Precision: 0.750 | Recall: 0.884 | F1: 0.812
Threshold: 0.7 | Precision: 0.749 | Recall: 0.861 | F1: 0.801
Threshold: 0.8 | Precision: 0.774 | Recall: 0.850 | F1: 0.810


In [19]:
import pandas as pd

error_df = pd.DataFrame({
    "text": X_test.values,
    "actual": y_test.values,
    "predicted": y_pred_bilstm,
    "probability": y_prob_bilstm
})

# False positives: genuine → predicted fraud
false_positives = error_df[
    (error_df["actual"] == 0) &
    (error_df["predicted"] == 1)
]

# False negatives: fraud → predicted genuine
false_negatives = error_df[
    (error_df["actual"] == 1) &
    (error_df["predicted"] == 0)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

False positives: 61
False negatives: 20


In [20]:
print("----- FALSE POSITIVE EXAMPLES -----")

for i, row in false_positives.head(3).iterrows():
    print("\nProbability:", round(row["probability"], 3))
    print(row["text"][:1000])

print("\n\n----- FALSE NEGATIVE EXAMPLES -----")

for i, row in false_negatives.head(3).iterrows():
    print("\nProbability:", round(row["probability"], 3))
    print(row["text"][:1000])

----- FALSE POSITIVE EXAMPLES -----

Probability: 0.97
Joomla Developer  Website development using Joomla 3 + jreviews + jomsocial, previous experience required  

Probability: 0.508
Organic & Analytical Chemist  SummaryThe Organic &amp; Analytical Chemist is responsible for development and upkeep of robust raw material specifications, MSDS and IMDS documentation for company products, compliance with GADSL requirements, chemical analysis and VOC characterization of raw materials and final products. The Organic &amp; Analytical Chemist will work with company Purchasing and our supply base to identify the appropriate Certificate of Analysis. As a secondary role, the Organic &amp; Analytical Chemist will provide analytical support to Product Development, Product Engineering, Product Testing, Manufacturing, and Quality.Essential Duties and Responsibilities         Improve existing company raw material specifications to emphasize critical parameters and identify acceptable tolerance limits 

In [21]:
uncertain_cases = error_df.copy()

uncertain_cases["distance_from_0.5"] = (
    uncertain_cases["probability"] - 0.5
).abs()

uncertain_cases = uncertain_cases.sort_values("distance_from_0.5")

for i, row in uncertain_cases.head(5).iterrows():
    print("\n--- UNCERTAIN CASE ---")
    print("Actual:", row["actual"])
    print("Probability:", round(row["probability"], 3))
    print("Predicted:", row["predicted"])
    print(row["text"][:1000])


--- UNCERTAIN CASE ---
Actual: 1
Probability: 0.499
Predicted: 0
Frame Stylist / Optician  If you enjoy sales and fashion, consider this position as a full-time Optical Frame Stylist in a private optometry practice in Newcastle, OK.  Experience is preferred, however we will consider an individual with a strong retail sales and/or fashion-related background. To apply for this position, please submit your application via this link: #URL_dc6a4e8df8c88cf7bb611c27fadf835b2ea6d40cec837463b39bb6ba9bca8852#?i=MTkz and select the Frame Stylist (Newcastle, Oklahoma) position from the Job Opening drop-down menu.We are an equal opportunity employer. Responsibilities include educating and assisting patients in selecting their glasses frames and lenses, taking patient glasses related measurements, presenting and dispensing glasses, maintaining glasses inventory, and communicating with doctors, staff, and patients. We offer a team-oriented work environment with benefits including: medical, vision, p

In [22]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_prob_val = model_bilstm.predict(
    X_val_pad,
    verbose=0
).ravel()

thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

print("Threshold | Precision | Recall | F1")
print("--------------------------------------")

for threshold in thresholds:
    y_pred_val = (y_prob_val >= threshold).astype(int)

    precision = precision_score(y_val, y_pred_val)
    recall = recall_score(y_val, y_pred_val)
    f1 = f1_score(y_val, y_pred_val)

    print(
        f"{threshold:.1f}       | "
        f"{precision:.3f}     | "
        f"{recall:.3f}  | "
        f"{f1:.3f}"
    )

Threshold | Precision | Recall | F1
--------------------------------------
0.2       | 0.566     | 0.835  | 0.674
0.3       | 0.603     | 0.820  | 0.695
0.4       | 0.640     | 0.806  | 0.713
0.5       | 0.671     | 0.806  | 0.732
0.6       | 0.701     | 0.791  | 0.743
0.7       | 0.721     | 0.763  | 0.741
0.8       | 0.770     | 0.748  | 0.759


In [23]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

FINAL_THRESHOLD = 0.8

y_pred_final = (y_prob_bilstm >= FINAL_THRESHOLD).astype(int)

print("Final Threshold:", FINAL_THRESHOLD)
print("Accuracy:", accuracy_score(y_test, y_pred_final))
print("Precision:", precision_score(y_test, y_pred_final))
print("Recall:", recall_score(y_test, y_pred_final))
print("F1:", f1_score(y_test, y_pred_final))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_final))

Final Threshold: 0.8
Accuracy: 0.9807046979865772
Precision: 0.7736842105263158
Recall: 0.8497109826589595
F1: 0.8099173553719008

Confusion Matrix:
[[3360   43]
 [  26  147]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      3403
           1       0.77      0.85      0.81       173

    accuracy                           0.98      3576
   macro avg       0.88      0.92      0.90      3576
weighted avg       0.98      0.98      0.98      3576

